# Simple RAG with PDF Files

This notebook demonstrates a simple Retrieval-Augmented Generation workflow using two PDF files:

- `BASEL.pdf`
- `COI.pdf`

The notebook first checks answers **without RAG**.

It then builds a RAG pipeline using:

```text
PDF
→ Load
→ Split into chunks
→ Create embeddings
→ Store in Chroma
→ Retrieve relevant chunks
→ LLM
→ Answer
```

Finally, the results are validated using a small question-and-answer file.

In [ ]:
# Install once:
# pip install langchain==0.2.17
# pip install langchain-community==0.2.19
# pip install langchain-openai==0.1.25
# pip install langchain-chroma==0.1.4
# pip install chromadb==0.5.5
# pip install pypdf pandas python-dotenv

## API Key

Create a `.env` file in the same folder:

```text
OPENAI_API_KEY=your_openai_api_key
```

The API key is read from the environment instead of being written directly in the notebook.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY is not configured.")
else:
    print("OPENAI_API_KEY is available.")

# Part 1 - Load the Validation Questions

The validation CSV contains:

- `question`
- `answer`
- `source`

The `answer` column contains the expected answer.

The `source` column tells us which PDF contains the answer.

In [ ]:
test_data = pd.read_csv("rag_validation_questions.csv")
test_data

# Part 2 - Check Without RAG

In this step, the LLM does not receive `BASEL.pdf` or `COI.pdf`.

The instruction asks the model not to guess when source documents are unavailable.

This creates a simple baseline for comparison.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## Step 1 - Create a simple function without RAG

In [ ]:
def answer_without_rag(question):
    prompt = f'''
You are answering questions that must be supported by provided reference documents.

No reference documents have been provided.

Do not use external or prior knowledge.
Do not guess.

If the answer cannot be verified from provided documents, respond exactly:

I don't know from the provided documents.

Question:
{question}
'''

    response = llm.invoke(prompt)
    return response.content

## Step 2 - Generate answers without RAG

In [ ]:
without_rag_predictions = []

for _, row in test_data.iterrows():
    predicted_answer = answer_without_rag(row["question"])

    without_rag_predictions.append({
        "question": row["question"],
        "expected_answer": row["answer"],
        "predicted_answer": predicted_answer
    })

without_rag_df = pd.DataFrame(without_rag_predictions)
without_rag_df

# Part 3 - Load the PDF Files

`PyPDFLoader` reads the PDF files page by page.

Both PDF documents are loaded into one list of documents.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    "BASEL.pdf",
    "COI.pdf"
]

pages = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    pdf_pages = loader.load()

    for page in pdf_pages:
        page.metadata["source_file"] = pdf_file

    pages.extend(pdf_pages)

print("Total pages loaded:", len(pages))

## Step 3 - Check the Loaded Pages

Each page contains:

- `page_content`
- `metadata`

The metadata contains the source PDF and page number.

In [ ]:
print(pages[0].page_content[:1000])
print()
print(pages[0].metadata)

# Part 4 - Split the PDF Text into Chunks

Large PDF pages are divided into smaller chunks.

This example uses:

```text
chunk_size = 1000
chunk_overlap = 150
```

The overlap keeps a small amount of text from the previous chunk.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(pages)

print("Total chunks:", len(chunks))

In [ ]:
print(chunks[0].page_content)
print()
print(chunks[0].metadata)

# Part 5 - Create Embeddings

Embeddings convert every text chunk into a numerical representation.

Chunks with similar meaning have similar vector representations.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

## Step 4 - Create One Sample Embedding

In [ ]:
sample_embedding = embeddings.embed_query(
    "What is the Liquidity Coverage Ratio?"
)

print("Embedding length:", len(sample_embedding))
print(sample_embedding[:10])

# Part 6 - Store the Chunks in Chroma

Chroma stores:

- document chunks,
- embeddings,
- metadata.

The vector database is used during retrieval.

In [ ]:
from langchain_chroma import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="rag_vector_db"
)

print("Vector database created.")

# Part 7 - Retrieval

The retriever searches the vector database for chunks related to the question.

In [ ]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 4}
)

## Step 5 - Test Retrieval

In [ ]:
question = "Which Article provides equality before law?"

relevant_docs = retriever.get_relevant_documents(question)

for i, doc in enumerate(relevant_docs, start=1):
    print("=" * 80)
    print("Document:", i)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print()
    print(doc.page_content[:800])

# Part 8 - Create the RAG Question Answering Chain

`RetrievalQA` performs two main operations:

1. retrieves relevant chunks;
2. sends those chunks with the question to the LLM.

`return_source_documents=True` also gives us the retrieved source documents.

In [ ]:
from langchain.chains import RetrievalQA

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

## Step 6 - Ask One Question with RAG

In [ ]:
query = "Which Article of the Constitution of India provides equality before law?"

result = rag_chain.invoke({"query": query})

print("Answer:")
print(result["result"])

## Step 7 - Display the Sources Used

In [ ]:
for doc in result["source_documents"]:
    print(
        "Source:",
        doc.metadata.get("source_file"),
        "| Page:",
        doc.metadata.get("page")
    )

# Part 9 - Generate RAG Predictions for All Questions

The same RAG chain is now executed for every validation question.

In [ ]:
rag_predictions = []

for _, row in test_data.iterrows():

    result = rag_chain.invoke({
        "query": row["question"]
    })

    sources = list({
        doc.metadata.get("source_file")
        for doc in result["source_documents"]
    })

    rag_predictions.append({
        "question": row["question"],
        "expected_answer": row["answer"],
        "expected_source": row["source"],
        "predicted_answer": result["result"],
        "retrieved_sources": ", ".join(sources)
    })

rag_df = pd.DataFrame(rag_predictions)
rag_df

# Part 10 - Validate Answer Accuracy

`QAEvalChain` uses an LLM to compare:

```text
Expected Answer
vs
Predicted Answer
```

The result is classified as `CORRECT` or `INCORRECT`.

In [ ]:
from langchain.evaluation.qa import QAEvalChain

qa_eval_chain = QAEvalChain.from_llm(llm)

## Step 8 - Prepare Data for Evaluation

In [ ]:
def prepare_eval_data(dataframe):
    examples = []
    predictions = []

    for _, row in dataframe.iterrows():
        examples.append({
            "question": row["question"],
            "answer": row["expected_answer"]
        })

        predictions.append({
            "question": row["question"],
            "result": row["predicted_answer"]
        })

    return examples, predictions

## Step 9 - Evaluate Without RAG

In [ ]:
without_examples, without_predictions = prepare_eval_data(
    without_rag_df
)

without_eval = qa_eval_chain.evaluate(
    without_examples,
    without_predictions,
    question_key="question",
    answer_key="answer",
    prediction_key="result"
)

without_eval

## Step 10 - Calculate Accuracy Without RAG

In [ ]:
def calculate_accuracy(eval_results):

    correct = 0

    for item in eval_results:
        grade = str(item.get("results", "")).upper()

        if "CORRECT" in grade and "INCORRECT" not in grade:
            correct += 1

    return correct / len(eval_results)

without_rag_accuracy = calculate_accuracy(without_eval)

print(
    "Without RAG Accuracy:",
    round(without_rag_accuracy * 100, 2),
    "%"
)

## Step 11 - Evaluate With RAG

In [ ]:
rag_examples, rag_prediction_list = prepare_eval_data(
    rag_df
)

rag_eval = qa_eval_chain.evaluate(
    rag_examples,
    rag_prediction_list,
    question_key="question",
    answer_key="answer",
    prediction_key="result"
)

rag_eval

## Step 12 - Calculate Accuracy With RAG

In [ ]:
rag_accuracy = calculate_accuracy(rag_eval)

print(
    "With RAG Accuracy:",
    round(rag_accuracy * 100, 2),
    "%"
)

# Part 11 - Citation / Source Validation

The source validation checks whether the expected PDF appears in the retrieved documents.

Example:

```text
Question belongs to COI.pdf
Retrieved source contains COI.pdf
→ Citation / Source Match = 1
```

In [ ]:
rag_df["source_match"] = rag_df.apply(
    lambda row:
        1 if row["expected_source"] in row["retrieved_sources"]
        else 0,
    axis=1
)

citation_accuracy = rag_df["source_match"].mean()

print(
    "Citation / Source Accuracy:",
    round(citation_accuracy * 100, 2),
    "%"
)

# Part 12 - Simple Groundedness Check

Groundedness checks whether the answer is supported by the retrieved context.

For each RAG answer:

```text
Question
+
Retrieved Context
+
Generated Answer
→ LLM Judge
→ GROUNDED / NOT_GROUNDED
```

In [ ]:
def check_groundedness(question, answer, source_documents):

    context = "\n\n".join(
        doc.page_content
        for doc in source_documents
    )

    prompt = f'''
Check whether the answer is supported by the context.

Return only one word:

GROUNDED
or
NOT_GROUNDED

Question:
{question}

Context:
{context}

Answer:
{answer}
'''

    response = llm.invoke(prompt)

    return response.content.strip()

## Step 13 - Calculate Groundedness

In [ ]:
groundedness_results = []

for _, row in test_data.iterrows():

    result = rag_chain.invoke({
        "query": row["question"]
    })

    groundedness = check_groundedness(
        row["question"],
        result["result"],
        result["source_documents"]
    )

    groundedness_results.append({
        "question": row["question"],
        "groundedness": groundedness
    })

groundedness_df = pd.DataFrame(
    groundedness_results
)

groundedness_df

In [ ]:
grounded_count = (
    groundedness_df["groundedness"]
    .str.upper()
    .eq("GROUNDED")
    .sum()
)

groundedness_score = (
    grounded_count /
    len(groundedness_df)
)

print(
    "Groundedness:",
    round(groundedness_score * 100, 2),
    "%"
)

# Part 13 - Retrieval Success

Retrieval success checks whether the correct PDF was retrieved for each question.

This is a simple retrieval metric.

In [ ]:
retrieval_success = rag_df["source_match"].mean()

print(
    "Retrieval Success:",
    round(retrieval_success * 100, 2),
    "%"
)

# Part 14 - Final Comparison

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Without RAG Accuracy",
        "With RAG Accuracy",
        "Citation / Source Accuracy",
        "Groundedness",
        "Retrieval Success"
    ],
    "Score": [
        without_rag_accuracy,
        rag_accuracy,
        citation_accuracy,
        groundedness_score,
        retrieval_success
    ]
})

summary["Percentage"] = (
    summary["Score"] * 100
).round(2)

summary

# Final Flow

## Without RAG

```text
Question
→ LLM
→ No reference documents
→ Answer / I don't know
→ Validation
```

## With RAG

```text
PDF Files
→ PyPDFLoader
→ RecursiveCharacterTextSplitter
→ OpenAIEmbeddings
→ Chroma
→ Retriever
→ RetrievalQA
→ Answer
→ Source Documents
→ Validation
```

## Metrics

The notebook uses five simple checks:

1. **Without RAG Accuracy**
2. **With RAG Accuracy**
3. **Citation / Source Accuracy**
4. **Groundedness**
5. **Retrieval Success**

The comparison shows the value of supplying relevant source context before asking the LLM to answer document-specific questions.